***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 4 章：可见度空间](4_0_introduction.ipynb)
    * 上一节：[4.4 可见度函数](4_4_the_visibility_function.ipynb)
    * 下一节：[4.5.2 提高 $uv$ 覆盖](4_5_2_uv_coverage_improving_your_coverage.ipynb)

***


导入标准模块。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


In [ ]:
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
HTML('../style/code_toggle.html')


## 4.5.1 $uv$ 覆盖：单条基线的 $uv$ 轨迹<a id='visibility:sec:uv_tracks'></a>

第 4.4 节把可见度写成天空亮度的空间频率表示。现在需要回答一个观测问题：干涉阵实际会在 `uv` 平面上测到哪些点。对一条物理基线而言，天线位置固定在地面上，但相位中心固定在天球上。随着地球自转，这条基线相对于相位中心的投影不断变化，于是在 `uv` 平面上画出一条轨迹。

这一节先只讨论单条基线。单条基线的轨迹看似简单，却已经包含了 `uv` 覆盖的核心直觉：观测时间越长，轨迹越长；源赤纬不同，椭圆形状不同；反向基线给出共轭采样点；若只覆盖很短的一段轨迹，成像时的脏波束会很差。


In [ ]:
fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)


def uvw_from_xyz(b_xyz_m, hour_angle_rad, dec_rad, wavelength_m=1.0):
    X, Y, Z = np.asarray(b_xyz_m, dtype=float) / wavelength_m
    H = np.asarray(hour_angle_rad)
    u = X * np.sin(H) + Y * np.cos(H)
    v = -X * np.sin(dec_rad) * np.cos(H) + Y * np.sin(dec_rad) * np.sin(H) + Z * np.cos(dec_rad)
    w = X * np.cos(dec_rad) * np.cos(H) - Y * np.cos(dec_rad) * np.sin(H) + Z * np.sin(dec_rad)
    return u, v, w


def visibility_point_sources(u, v, sources):
    vis = np.zeros_like(np.asarray(u, dtype=float), dtype=complex)
    for l0, m0, flux in sources:
        vis += flux * np.exp(-2j * np.pi * (u * l0 + v * m0))
    return vis

# Figure 4.5.1: a single uv track.
wavelength = 0.21
baseline = np.array([85.0, 135.0, 18.0])
dec = np.deg2rad(-30.0)
ha_h = np.linspace(-6.0, 6.0, 700)
ha_rad = ha_h * np.pi / 12.0
u, v, w = uvw_from_xyz(baseline, ha_rad, dec, wavelength)

fig, axes = plt.subplots(1, 2, figsize=(12.2, 5.1), constrained_layout=True)
axes[0].plot(u, v, color='#2b6cb0', lw=2.0, label=r'$\mathbf{b}_{12}$')
axes[0].plot(-u, -v, color='#c53030', lw=1.5, ls='--', label=r'$\mathbf{b}_{21}$')
axes[0].scatter([u[len(u)//2]], [v[len(v)//2]], color='black', s=35, zorder=4, label='transit')
axes[0].axhline(0, color='0.78', lw=0.8)
axes[0].axvline(0, color='0.78', lw=0.8)
axes[0].set_aspect('equal', adjustable='box')
axes[0].set_xlabel('u [wavelengths]')
axes[0].set_ylabel('v [wavelengths]')
axes[0].set_title('Projected baseline in the uv plane')
axes[0].legend(frameon=False, loc='upper right')

axes[1].plot(ha_h, u, lw=1.8, label='u')
axes[1].plot(ha_h, v, lw=1.8, label='v')
axes[1].plot(ha_h, w, lw=1.8, label='w')
axes[1].axvline(0, color='0.75', lw=0.9, ls=':')
axes[1].set_xlabel('hour angle H [h]')
axes[1].set_ylabel('coordinate [wavelengths]')
axes[1].set_title('uvw coordinates during a 12 h track')
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False, ncol=3)
fig.savefig(fig_dir / 'uv_track_single_baseline.png', dpi=180)
plt.close(fig)

# Figure 4.5.2: declination dependence.
fig, ax = plt.subplots(figsize=(7.0, 6.4))
for dec_deg, color in [(-70, '#1f77b4'), (-30, '#2ca02c'), (0, '#9467bd'), (45, '#d62728')]:
    uu, vv, ww = uvw_from_xyz(baseline, ha_rad, np.deg2rad(dec_deg), wavelength)
    ax.plot(uu, vv, lw=1.8, color=color, label=fr'$\delta={dec_deg:+d}^\circ$')
ax.axhline(0, color='0.78', lw=0.8)
ax.axvline(0, color='0.78', lw=0.8)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('u [wavelengths]')
ax.set_ylabel('v [wavelengths]')
ax.set_title('The same physical baseline gives different tracks for different declinations')
ax.grid(alpha=0.2)
ax.legend(frameon=False, loc='upper right')
fig.savefig(fig_dir / 'uv_track_declination_dependence.png', dpi=180)
plt.close(fig)

# Figure 4.5.3: sampled visibility along a track.
sources = [(-0.055, 0.018, 1.0), (0.070, -0.030, 0.55), (0.015, 0.060, 0.32)]
vis = visibility_point_sources(u, v, sources)
fig, axes = plt.subplots(2, 2, figsize=(12.0, 7.5), constrained_layout=True)
axes[0, 0].scatter([s[0] for s in sources], [s[1] for s in sources], s=[220*s[2] for s in sources], color='#c53030')
axes[0, 0].axhline(0, color='0.78', lw=0.8)
axes[0, 0].axvline(0, color='0.78', lw=0.8)
axes[0, 0].set_xlim(-0.11, 0.11)
axes[0, 0].set_ylim(-0.09, 0.09)
axes[0, 0].set_aspect('equal', adjustable='box')
axes[0, 0].set_xlabel('l')
axes[0, 0].set_ylabel('m')
axes[0, 0].set_title('Toy sky model')
axes[0, 1].plot(u, v, color='#2b6cb0', lw=1.7)
axes[0, 1].set_aspect('equal', adjustable='box')
axes[0, 1].set_xlabel('u [wavelengths]')
axes[0, 1].set_ylabel('v [wavelengths]')
axes[0, 1].set_title('Sampled uv track')
axes[1, 0].plot(ha_h, vis.real, lw=1.7, label='Re V')
axes[1, 0].plot(ha_h, vis.imag, lw=1.7, label='Im V')
axes[1, 0].axhline(0, color='0.78', lw=0.8)
axes[1, 0].set_xlabel('hour angle H [h]')
axes[1, 0].set_ylabel('visibility')
axes[1, 0].set_title('Complex visibility along the track')
axes[1, 0].legend(frameon=False)
axes[1, 1].plot(ha_h, np.abs(vis), lw=1.7, label='amplitude')
axes[1, 1].plot(ha_h, np.unwrap(np.angle(vis)), lw=1.7, label='unwrapped phase')
axes[1, 1].set_xlabel('hour angle H [h]')
axes[1, 1].set_title('Amplitude and phase time series')
axes[1, 1].legend(frameon=False)
fig.savefig(fig_dir / 'uv_track_visibility_samples.png', dpi=180)
plt.close(fig)


### 4.5.1.1 `uvw` 坐标随时角变化<a id='visibility:sec:uvw_track_equations'></a>

第 4.2 节给出的 $XYZ\rightarrow uvw$ 变换可以直接写成时角的函数。设物理基线在 $XYZ$ 参考架中的分量为 $(X,Y,Z)$，相位中心赤纬为 $\delta_0$，时角为 $H$，并且所有长度已经除以波长，则

$$
u = X\sin H + Y\cos H,
$$

$$
v = -X\sin\delta_0\cos H + Y\sin\delta_0\sin H + Z\cos\delta_0,
$$

$$
w = X\cos\delta_0\cos H - Y\cos\delta_0\sin H + Z\sin\delta_0.
$$

这些式子说明，地球自转通过 $H$ 进入投影基线。只要 $H$ 变化，$(u,v,w)$ 就变化；把 $u(H)$ 与 $v(H)$ 画出来，就得到该基线在 `uv` 平面上的轨迹。反向基线 $\mathbf{b}_{21}=-\mathbf{b}_{12}$ 会给出 $(-u,-v,-w)$，因此对于实天空亮度，可见度满足

$$
V(-u,-v)=V^*(u,v).
$$


![单条基线的 uv 轨迹](figures/uv_track_single_baseline.png)

**图 4.5.1** 单条物理基线在 12 小时跟踪中的 `uv` 轨迹及其 `uvw` 坐标随时角的变化。反向基线给出关于原点对称的共轭采样点。


### 4.5.1.2 轨迹形状取决于源赤纬<a id='visibility:sec:uv_track_declination'></a>

同一条物理基线观测不同赤纬的源时，`uv` 轨迹并不相同。原因是相位中心方向改变后，局部切平面也随之改变；同一条地面基线投影到这个切平面上的方式发生变化。一般情况下，轨迹在 `uv` 平面中呈椭圆弧。若观测时间覆盖完整恒星日，就会得到完整闭合轨迹；实际观测通常只覆盖其中一段。

几个特殊情形很有用。极地阵列观测北天极附近源时，基线在源方向下几乎做纯旋转，轨迹接近圆。赤道阵列观测天赤道源时，部分基线投影会退化成较接近直线的轨迹。东西向阵列具有额外对称性，`uv` 轨迹完全落在一个平面内，这使得早期综合孔径理论分析格外清晰。


![赤纬对 uv 轨迹的影响](figures/uv_track_declination_dependence.png)

**图 4.5.2** 同一条物理基线观测不同赤纬源时得到的 `uv` 轨迹。源赤纬改变了相位中心切平面的方向，因此改变了投影轨迹。


### 4.5.1.3 沿轨迹采样可见度函数<a id='visibility:sec:visibility_along_track'></a>

`uv` 轨迹不是几何装饰，而是可见度函数被采样的位置。给定一个天空模型和一条轨迹，观测得到的时间序列可以写成

$$
V(t)=V\big(u(t),v(t)\big).
$$

因此，相关器随时间输出的复数序列，本质上是在连续可见度函数上沿一条曲线取样。若天空只有中心点源，沿轨迹的可见度幅度几乎不变；若天空有多个偏心源或扩展结构，实部、虚部、幅度和相位都会随时角变化。复杂源的真实观测数据之所以看起来难以直接解释，正是因为每条基线都在可见度函数的不同路径上取样。


![沿 uv 轨迹采样可见度](figures/uv_track_visibility_samples.png)

**图 4.5.3** 一个三点源天空模型沿单条 `uv` 轨迹产生的可见度时间序列。时间轴上的实部、虚部、幅度和相位变化，都是同一条轨迹穿过可见度函数的结果。


### 4.5.1.4 本节要点

单条物理基线在地面上近似固定，但相对于相位中心的投影会随地球自转改变。以波长为单位的投影坐标 $(u,v,w)$ 由基线、源赤纬和时角共同决定；在 `uv` 平面中，这些坐标通常画出椭圆弧。观测数据就是沿这些轨迹取得的可见度样本。下一节将从单条基线推广到多天线阵列，并讨论怎样通过更多基线、更长观测时间和更宽频率覆盖改善 `uv` 覆盖。


***

* 下一节：[4.5.2 提高 $uv$ 覆盖](4_5_2_uv_coverage_improving_your_coverage.ipynb)
